# Fundamentals of RAG systems

## 1) Setting up the development environment

### Set up Asyncio

In [6]:
import nest_asyncio2

nest_asyncio2.apply()

### Set up the Qdrant vector database

In [ ]:
import qdrant_client

collection_name = "chat_with_docs"

client = qdrant_client.QdrantClient(
    host="localhost",
    port=6333
)

/home/francisco_bneto/gen-ai-formation/venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


## 2) Read the documents

In [19]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PyMuPDFReader

input_dir_path = "./docs"

loader = SimpleDirectoryReader(
    input_dir=input_dir_path,
    required_exts=[".pdf"],
    recursive=True
)

docs = loader.load_data()

print(f"Directory: {input_dir_path} | Type: {type(docs)} | Number of docs: {len(docs)}")

Directory: ./docs | Type: <class 'list'> | Number of docs: 32


## 3) Create a function to index data

In [21]:
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core import VectorStoreIndex, ServiceContext, StorageContext

def create_index(documents):
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name
    )

    storage_context = StorageContext.from_defaults(vector_store=vector_store)

    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context
    )

    return index

## 4) Load the embedding model and index data

In [23]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-large-en-v1.5",
    trust_remote_code=True
)

Settings.embed_model = embed_model

index = create_index(docs)

2026-07-02 14:58:39,442 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 14:58:39,917 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-07-02 14:58:40,507 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 14:58:40,972 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-02 14:58:40,978 - INFO - Loading SentenceTransformer model from BAAI/bge-large-en-v1.5.
2026-07-02 14:58:41,784 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 3

## 5) Load the LLM

In [ ]:
from llama_index.llms.ollama import Ollama

llm_model = Ollama(
    model="llama3.2:1b",
    request_timeout=120.0
)

Settings.llm = llm_model

## 6) Define the prompt template

In [25]:
from llama_index.core import PromptTemplate

template = """
Context information is below:
==============================================
{contexto_str}
==============================================
Given the context information above I want you to think step by step to answer the query in a crisp manner, incase you don't know the answer say 'I don't know!'

Query: {query_str}

Answer:"""

qa_prompt_tmpl = PromptTemplate(template)

In [29]:
print(qa_prompt_tmpl.template)


Context information is below:
{contexto_str}
Given the context information above I want you to think step by step to answer the query in a crisp manner, incase you don't know the answer say 'I don't know!'

Query: {query_str}

Answer:


## 7) Reranking

In [30]:
from llama_index.core.postprocessor import SentenceTransformerRerank

rerank = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",
    top_n=3
)

2026-07-02 15:27:45,267 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 15:27:45,865 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L2-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-07-02 15:27:45,888 - INFO - No modules.json found for cross-encoder/ms-marco-MiniLM-L-2-v2, initializing a new CrossEncoder model.
2026-07-02 15:27:46,605 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-2-v2 "HTTP/1.1 307 Temporary Redirect"
2026-07-02 15:27:46,857 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L2-v2 "HTTP/1.1 200 OK"
2026-07-02 15:27:47,113 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-2-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 15:27:47,730 - INFO - HTTP Request: HEAD https://hugg

## 8) Query the document

In [31]:
query_engine = index.as_query_engine(
    similarity_top_k=10,
    node_postprocessors=[rerank]
)

query_engine.update_prompts(
    {
        "response_synthesizer:text_qa_template": qa_prompt_tmpl
    }
)

response = query_engine.query("What exactly is DSPy?")

2026-07-02 16:14:34,999 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"
2026-07-02 16:14:40,708 - INFO - HTTP Request: POST http://localhost:6333/collections/chat_with_docs/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]
2026-07-02 16:15:09,173 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 500 Internal Server Error"


ResponseError: llama-server process has terminated: ggml_aligned_malloc: insufficient memory (attempted to allocate 14336.00 MB)
ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 15032385536
alloc_tensor_range: failed to allocate CPU buffer of size 15032385536
llama_init_from_model: failed to initialize the context: failed to allocate buffer for kv cache: ggml_aligned_malloc: insufficient memory (attempted to allocate 14336.00 MB)
ggml_backend_cpu_buffer_type_alloc_buffer: failed to allocate buffer of size 15032385536
alloc_tensor_range: failed to allocate CPU buffer of size 15032385536
llama_init_from_model: failed to initialize the context: failed to allocate buffer for kv cache (status code: 500)